# Multi-Task-LSTM

**Hypothese:** Ein LSTM kann die Reihenfolge der hyperspektralen Bänder nutzen und damit Crop und Stage mindestens so gut trennen wie ein tabellarisches Multi-Task-Modell.

Das Modell verarbeitet die skalierten Spektralbänder als Sequenz; AEZ und Month bleiben Kontextmerkmale außerhalb der Wellenlängenachse. Die Suche nutzt ausschließlich die gemeinsamen Trainingsfolds. Erst der per CV bestimmte Gewinner wird einmal auf der Validierung bewertet.

In [ ]:
%load_ext autoreload
%autoreload 2

from IPython.display import display

from awp2.experiment import RunConfig, TuneConfig, run, tune
from awp2.models import MultiTaskLSTMClassifier
from awp2.preprocessing import PreprocessingConfig

PREPROCESSING = PreprocessingConfig(use_spectral_standard_scale=True)
HIDDEN_UNITS = [16, 32]
LAYERS = [1, 2]
DROPOUT = 0.2
MAX_EPOCHS = 50

model = MultiTaskLSTMClassifier(dropout=DROPOUT, max_epochs=MAX_EPOCHS)
model.get_params(deep=True)

## Hyperparameter suchen

Die vier Kandidaten unterscheiden die Breite und Tiefe des LSTM. Dropout und die maximale Epochenzahl bleiben konstant. Die Auswahl erfolgt nur über die mittlere kombinierte BAcc der fünf Trainingsfolds.

In [ ]:
tuned = tune(
    model,
    TuneConfig(
        name="multi_task_lstm_search",
        approach="multi-task",
        description=(
            "CV-only: Multi-Task-LSTM mit skalierten Spektralbändern; "
            "Breite und Anzahl der LSTM-Schichten vergleichen."
        ),
        preprocessing=PREPROCESSING,
        param_grid={
            "hidden_units": HIDDEN_UNITS,
            "layers": LAYERS,
        },
    ),
)

display(tuned.candidates.sort_values("rank"))
{
    "best_cv_score": tuned.best_score,
    "best_params": tuned.best_params,
    "tuning_run": tuned.run_id,
}

## Einmalige Holdout-Bewertung

Diese Zelle wird genau einmal nach der CV-Entscheidung ausgeführt. Sie nutzt keine Validierungsmetrik zur Auswahl weiterer Hyperparameter.

In [ ]:
result = run(
    tuned.best_model,
    RunConfig(
        name="multi_task_lstm",
        approach="multi-task",
        description=(
            "Bester Multi-Task-LSTM aus multi_task_lstm_search, "
            "einmal auf der gemeinsamen Validierung bewertet."
        ),
        preprocessing=PREPROCESSING,
        tuning_run=tuned.run_id,
    ),
)

result.metrics.model_dump()

## Auswertung vorbereiten

Die Tabellen und Abbildungen zeigen die CV-Streuung, die Confusion Matrices, die Klassenrecalls und die häufigsten Verwechslungen. Sie werden erst nach dem Ausführen der beiden vorherigen Codezellen erzeugt.

In [ ]:
from awp2.evaluation import plot_confusion_matrices
from awp2.plots import plot_class_recall, plot_top_confusions, plot_tuning_candidates

tuning_ax = plot_tuning_candidates(
    tuned.candidates,
    color_by="layers",
    validation_score=result.metrics.bacc_combined,
)
tuning_ax.set_title("Multi-Task-LSTM: CV-Score der Kandidaten")
display(tuning_ax.figure)

for figure in (
    plot_confusion_matrices(result.y_val, result.y_pred),
    plot_class_recall(result.y_val, result.y_pred),
    plot_top_confusions(result.y_val, result.y_pred),
):
    display(figure)

{
    "tuning_run": tuned.run_id,
    "evaluation_run": result.run_id,
    "metrics": result.metrics.model_dump(),
    "invalid_combinations": result.metrics.invalid_combinations,
}